# 📊 EDA Report – Quant Analyst
**Workflow**: Load OHLCV → Rich Describe → Distributions → Rolling Volatility → Export HTML Report

---

In [1]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from scipy.special import erfinv

session = AnalystSession(role=RoleContext.ANALYST)
session.info()

## 1. Load Data

In [2]:
SYMBOL = "BTC-USD"
TIMEFRAME = "1d"

try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME, days=365)
    if len(df) < 50:  # Too few rows for rolling features — use synthetic data
        raise ValueError(f"Insufficient data: only {len(df)} rows")
    print(f"Loaded {len(df)} rows from datalake")
except Exception as e:
    print(f"Falling back to synthetic data: {e}")
    df = session.sample_ohlcv(symbol="BTC", days=730)
    print(f"Using synthetic data: {len(df)} rows")

df.head(5)

timestamp,open,high,low,close,volume,symbol,tf
datetime[μs],f64,f64,f64,f64,f64,str,str
2026-03-09 00:00:00,65970.56,69543.0,65820.93,68432.17,13303.956154,"""BTC-USD""","""1d"""
2026-03-10 00:00:00,68432.16,71800.0,68383.5,69960.83,13322.433999,"""BTC-USD""","""1d"""
2026-03-11 00:00:00,69960.83,71358.48,68980.74,70208.0,9851.728705,"""BTC-USD""","""1d"""
2026-03-12 00:00:00,70208.03,70819.84,69180.01,70531.56,9708.420467,"""BTC-USD""","""1d"""
2026-03-13 00:00:00,70530.02,73968.0,70398.01,70944.33,12304.216338,"""BTC-USD""","""1d"""


## 2. Rich Descriptive Statistics

In [3]:
df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 14, 21])
df = df.drop_nulls()  # Drop NaNs from rolling windows before analysis
assert len(df) > 0, "DataFrame is empty after feature engineering — check data source"
stats_df = session.rich_describe_table(df)
stats_df

## 3. Returns Distribution

In [ ]:
returns = df['returns'].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor='#0f1117')
for ax in axes:
    ax.set_facecolor('#0f1117')
    for spine in ax.spines.values():
        spine.set_color('#334155')
    ax.tick_params(colors='#94a3b8')

# Histogram + KDE
axes[0].hist(returns, bins=80, color='#38bdf8', alpha=0.7, density=True, label='Returns')
mu, sigma = returns.mean(), returns.std()
x = np.linspace(returns.min(), returns.max(), 300)
axes[0].plot(x, np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi)), 'r--', linewidth=1.5, label='Normal')
axes[0].set_title('Returns Distribution', color='#e2e8f0')
axes[0].legend(facecolor='#1e293b', labelcolor='#e2e8f0')

# QQ-plot approximation
sorted_r = np.sort(returns)
n = len(sorted_r)
quantiles = np.array([(i+0.5)/n for i in range(n)])
normal_q = mu + sigma * np.sqrt(2) * np.array([erfinv(2*p-1) for p in quantiles])
axes[1].scatter(normal_q, sorted_r, s=2, color='#38bdf8', alpha=0.5)
axes[1].plot([normal_q[0], normal_q[-1]], [normal_q[0], normal_q[-1]], 'r--', linewidth=1)
axes[1].set_title('QQ Plot vs Normal', color='#e2e8f0')
axes[1].set_xlabel('Theoretical Quantiles', color='#94a3b8')
axes[1].set_ylabel('Sample Quantiles', color='#94a3b8')

plt.tight_layout()
plt.show()
dist_fig = fig

## 4. Rolling Volatility & Price

In [ ]:
fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), facecolor='#0f1117', sharex=True)
for ax in [ax1, ax2]:
    ax.set_facecolor('#0f1117')
    for spine in ax.spines.values(): spine.set_color('#334155')
    ax.tick_params(colors='#94a3b8')
    ax.grid(color='#1e293b', linestyle='--', linewidth=0.5)

close = df['close'].to_numpy()
ax1.plot(close, color='#38bdf8', linewidth=1.2)
ax1.set_ylabel('Close Price', color='#94a3b8')
ax1.set_title(f'{SYMBOL} Price & Rolling Volatility', color='#e2e8f0')

if 'vol_21' in df.columns:
    vol21 = df['vol_21'].to_numpy()
    ax2.fill_between(range(len(vol21)), vol21, alpha=0.6, color='#f472b6')
    ax2.set_ylabel('Rolling Vol (21)', color='#94a3b8')

plt.tight_layout()
plt.show()
vol_fig = fig2

## 5. Export HTML Report

In [ ]:
out_path = session.export_report(
    title=f"EDA Report – {SYMBOL} {TIMEFRAME}",
    sections={
        "Overview": f"Symbol: {SYMBOL} | Timeframe: {TIMEFRAME} | Rows: {len(df)}",
        "Descriptive Statistics": stats_df,
        "Returns Distribution": dist_fig,
        "Price & Rolling Volatility": vol_fig,
    },
    path=f"reports/{SYMBOL.replace('-','_')}_EDA.html",
)
print(f"✅ Report saved: {out_path}")